# 11 — Measure key events


Measure positive, negative, peak-to-peak, and reduced pressure for the key
arrivals using the canonical Notebook 02 moving-median-baseline-corrected
pressure Stream. The short local baseline windows used by the measurement
helper are retained only to remove any small residual constant offset.


This compact notebook provides named-event measurements used in the manuscript
and figure annotations. It remains separate from Notebook 10 because these
windows are manually defined for a few physically important arrivals rather
than derived from the full event catalogue.


In [ ]:
from pathlib import Path
import json
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime

from modules import project_config as config

DERIVED_DIR = config.DERIVED_DIR
FIGURE_DIR = config.FIGURE_DIR

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [ ]:
from obspy import read
from event_measurements import (
    measure_reduced_pressures_in_window,
    pressure_results_for_paper,
)

ANALYSIS_CONFIG_FILE = config.OUTPUT_DIR / "02_analysis_configuration.json"
GEOMETRY_FILE = config.OUTPUT_DIR / "02_bchh_geometry.csv"

if not ANALYSIS_CONFIG_FILE.exists():
    raise FileNotFoundError(ANALYSIS_CONFIG_FILE)
if not GEOMETRY_FILE.exists():
    raise FileNotFoundError(GEOMETRY_FILE)

analysis_config = json.loads(ANALYSIS_CONFIG_FILE.read_text())
baseline_products = analysis_config.get("baseline_removed_streams", {})
BASELINE_STREAM_FILE = Path(
    baseline_products.get(
        "pickle",
        DERIVED_DIR / "bchh_corrected_moving_median_baseline_removed.pkl",
    )
).expanduser()

if not BASELINE_STREAM_FILE.exists():
    raise FileNotFoundError(
        "Run Notebook 02 to create the baseline-corrected Stream: "
        f"{BASELINE_STREAM_FILE}"
    )

st_corr = read(str(BASELINE_STREAM_FILE), format="PICKLE")
geometry = pd.read_csv(GEOMETRY_FILE)
SENSOR_DISTANCES_M = (
    geometry.loc[geometry["channel"].str.startswith("HD")]
    .set_index("channel")["distance_m"]
    .astype(float)
    .to_dict()
)

print("Pressure source:", BASELINE_STREAM_FILE)


## Measurement windows

In [ ]:

EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.080")

event_windows = [
    {
        "event": "Initial second-stage failure",
        "start": EXPLOSION_TIME + 3.0 - 0.08,
        "end": EXPLOSION_TIME + 5.0 - 0.08,
        "baseline_start_s": 0.10,
        "baseline_end_s": 0.80,
        "signal_start_s": 0.85,
        "signal_end_s": 1.40,
    },
    {
        "event": "Principal explosion",
        "start": EXPLOSION_TIME + 6.0 - 0.08,
        "end": EXPLOSION_TIME + 9.0 - 0.08,
        "baseline_start_s": 0.10,
        "baseline_end_s": 0.75,
        "signal_start_s": 0.75,
        "signal_end_s": 2.50,
    },
    {
        "event": "Capsule pulse 1",
        "start": UTCDateTime("2016-09-01T13:07:28.30"),
        "end": UTCDateTime("2016-09-01T13:07:28.75"),
        "baseline_start_s": 0.00,
        "baseline_end_s": 0.12,
        "signal_start_s": 0.12,
        "signal_end_s": 0.45,
    },
    {
        "event": "Capsule pulse 2",
        "start": UTCDateTime("2016-09-01T13:07:28.85"),
        "end": UTCDateTime("2016-09-01T13:07:29.25"),
        "baseline_start_s": 0.00,
        "baseline_end_s": 0.10,
        "signal_start_s": 0.10,
        "signal_end_s": 0.40,
    },
]


In [ ]:

all_results = []
event_streams = {}

for spec in event_windows:
    event_stream, result = measure_reduced_pressures_in_window(
        st_corr,
        spec["start"],
        spec["end"],
        SENSOR_DISTANCES_M,
        reference_distance_m=1000.0,
        event_name=spec["event"],
        baseline_start_s=spec["baseline_start_s"],
        baseline_end_s=spec["baseline_end_s"],
        signal_start_s=spec["signal_start_s"],
        signal_end_s=spec["signal_end_s"],
    )
    event_streams[spec["event"]] = event_stream
    all_results.append(result)

key_event_pressures = pd.concat(all_results, ignore_index=True)
display(pressure_results_for_paper(key_event_pressures))
key_event_pressures.to_csv(
    DERIVED_DIR / "key_event_pressure_measurements.csv",
    index=False,
)
